In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
words = open('../data/makemore/names.txt', 'r').read().splitlines()
words[:10]

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

In [ ]:
VOCAB_SIZE = 27

In [ ]:
def train_with_backprop(
    xenc: torch.Tensor,
    ys: torch.Tensor,
    generator: torch.Generator,
    w_size: tuple,
    samples_num: int | None = None,
    epoch: int | None = None,
    lr: float | None = None,
    l2: float | None = None,
    verbose: bool = False,
) -> torch.Tensor:
    _num = ys.nelement() if samples_num is None else samples_num
    _epoch = 200 if epoch is None else epoch
    _lr = 50 if lr is None else lr
    _l2 = 0.01 if l2 is None else l2

    W = torch.randn(w_size, generator=generator, requires_grad=True)

    for i in range(_epoch):
        # Forward pass
        logits = xenc @ W
        # counts = logits.exp()
        # probs = counts / counts.sum(1, keepdim=True)
        # loss = -probs[torch.arange(_num), ys].log().mean() + _l2 * (W**2).mean()
        loss = F.cross_entropy(logits, ys)  # It is more stable (no overflow / underflow)

        if verbose:
            print(i, loss.item())

        # Backward pass
        W.grad = None
        loss.backward()

        # Update weights
        W.data += -_lr * W.grad

    return W


def eval_on_holdout(
    xenc: torch.Tensor,
    ys: torch.Tensor,
    weights: torch.Tensor,
    samples_num: int | None = None,
    verbose: bool = False,
) -> float:
    _num = ys.nelement() if samples_num is None else samples_num

    logits = xenc @ weights
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(_num), ys].log().mean()
    if verbose:
        print(f'{loss.item():.4f}')

    return loss

# 0 Bigram model again

To compare and use for inference as a part of threegram model

In [ ]:
bg = torch.Generator().manual_seed(2147483647)

# Bigrams dataset
bxs, bys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        bxs.append(ix1)
        bys.append(ix2)

bxs = torch.tensor(bxs)
bys = torch.tensor(bys)

# Getting train/dev/test splits
total_rows = bys.size(0)
rnd_idxs = torch.randperm(total_rows, generator=bg)
train_idxs = rnd_idxs[:int(total_rows * 0.8)]
dev_idxs = rnd_idxs[len(train_idxs): int(total_rows * 0.9)]
test_idxs = rnd_idxs[(len(train_idxs) + len(dev_idxs)):]
train_bxs, train_bys = bxs[train_idxs], bys[train_idxs]
dev_bxs, dev_bys = bxs[dev_idxs], bys[dev_idxs]
test_bxs, test_bys = bxs[test_idxs], bys[test_idxs]

bnum = bys.nelement()
print(f'Number of examples for bigrams: {bnum}')

# BW = torch.randn((27, 27), generator=bg, requires_grad=True)

bxenc = F.one_hot(bxs, num_classes=27).float()

train_bxenc = F.one_hot(train_bxs, num_classes=27).float()
dev_bxenc = F.one_hot(dev_bxs, num_classes=27).float()
test_bxenc = F.one_hot(test_bxs, num_classes=27).float()

In [ ]:
# Bigrams training (on full dataset)
BW = train_with_backprop(bxenc, bys, bg, (VOCAB_SIZE, VOCAB_SIZE), samples_num=bnum, epoch=200, lr=50, l2=0.01, verbose=True)

In [ ]:
# Bigrams full evaluation (on full dataset)
eval_on_holdout(bxenc, bys, BW, samples_num=bnum, verbose=True)

In [ ]:
# Bigrams training
BW = train_with_backprop(train_bxenc, train_bys, bg, (VOCAB_SIZE, VOCAB_SIZE), epoch=200, lr=50, l2=0.01, verbose=True)

In [ ]:
# Bigrams evaluation (on dev)
eval_on_holdout(dev_bxenc, dev_bys, BW, verbose=True)

In [ ]:
# Bigrams evaluation (on test)
eval_on_holdout(test_bxenc, test_bys, BW, verbose=True)

# 1 Threegrams

In [ ]:
# DATASET
g = torch.Generator().manual_seed(2147483647)

xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        idx1 = stoi[ch1]
        idx2 = stoi[ch2]
        idx3 = stoi[ch3]
        xs.append([idx1, idx2])
        ys.append(idx3)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

# Getting train/dev/test splits
total_rows = ys.size(0)
rnd_idxs = torch.randperm(total_rows, generator=g)
train_idxs = rnd_idxs[:int(total_rows * 0.8)]
dev_idxs = rnd_idxs[len(train_idxs): int(total_rows * 0.9)]
test_idxs = rnd_idxs[(len(train_idxs) + len(dev_idxs)):]
train_xs, train_ys = xs[train_idxs, :], ys[train_idxs]
dev_xs, dev_ys = xs[dev_idxs, :], ys[dev_idxs]
test_xs, test_ys = xs[test_idxs, :], ys[test_idxs]

num = ys.nelement()

print(f'Number of examples for threegrams {num}')

# We want to concatenate two encoded vectors into a single 2x longer vector
# W = torch.randn((VOCAB_SIZE * 2, VOCAB_SIZE), generator=g, requires_grad=True)

xenc = F.one_hot(xs, num_classes=VOCAB_SIZE).float()
# Concatenate two last axes into the single one keeping the symbols order
xenc = xenc.reshape(xenc.shape[0], -1)

train_xenc = F.one_hot(train_xs, num_classes=VOCAB_SIZE).float(); train_xenc = train_xenc.reshape(train_xenc.shape[0], -1)
dev_xenc = F.one_hot(dev_xs, num_classes=VOCAB_SIZE).float(); dev_xenc = dev_xenc.reshape(dev_xenc.shape[0], -1)
test_xenc = F.one_hot(test_xs, num_classes=VOCAB_SIZE).float(); test_xenc = test_xenc.reshape(test_xenc.shape[0], -1)

In [ ]:
# TRAINING (on full dataset)
W = train_with_backprop(xenc, ys, g, (VOCAB_SIZE * 2, VOCAB_SIZE), samples_num=num, epoch=200, lr=50, l2=0.01, verbose=True)

In [ ]:
# Threegrams evaluation (on full dataset)
eval_on_holdout(xenc, ys, W, samples_num=num, verbose=True)

In [ ]:
# TRAINING
# We want to concatenate two encoded vectors into a single 2x longer vector, so the rows number is VOCAB_SIZE * 2
W = train_with_backprop(train_xenc, train_ys, g, (VOCAB_SIZE * 2, VOCAB_SIZE), epoch=200, lr=50, l2=0.01, verbose=True)

In [ ]:
# Threegrams evaluation (on dev)
eval_on_holdout(dev_xenc, dev_ys, W, verbose=True)

In [ ]:
# Threegrams evaluation (on test)
eval_on_holdout(test_xenc, test_ys, W, verbose=True)

In [ ]:
# INFERENCE (using bigram model to generate the first character)
infgen = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = ['.']  # Set the beginning character as default

    benc = F.one_hot(torch.tensor([0]), num_classes=VOCAB_SIZE).float()

    # Bigram model forward pass
    blogits = benc @ BW
    bcounts = blogits.exp()
    bprobs = bcounts / bcounts.sum(1, keepdim=True)
    # Bigram character sampling
    idx = torch.multinomial(bprobs, num_samples=1, replacement=True, generator=infgen).item()
    out.append(itos[idx])
    if idx == 0:  # unlikely, but let's assume this
        print(''.join(out).strip('.'))
        continue

    # Generate the rest of the word
    while True:
        enc = F.one_hot(torch.tensor([stoi[out[-2]], idx]), num_classes=VOCAB_SIZE).float()
        enc = enc.reshape(1, -1)

        # Threegram model forward pass
        logits = enc @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdim=True)
        # Threegram character sampling
        idx = torch.multinomial(probs, num_samples=1, replacement=True, generator=infgen).item()
        out.append(itos[idx])
        if idx == 0:
            break

    gen_word = ''.join(out)
    print(gen_word.strip('.'))